# Datamart checks

## 1. Start Spark and load the files

In [1]:
import pyspark
import pyspark.sql.functions as F
from pyspark.sql.functions import col
import os
import pandas as pd

spark = pyspark.sql.SparkSession.builder.appName("dev").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

DATA_DIR = "../data"
DATAMART = "../datamart"

SOURCES = {
    "attributes": "features_attributes.csv",
    "financials": "features_financials.csv",
    "clickstream": "feature_clickstream.csv",
    "loan_daily": "lms_loan_daily.csv",
}

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 08:10:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/28 08:10:03 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 2. Bronze holds exactly the raw data

In [2]:
for name in SOURCES:
    files = sorted(os.listdir(f"{DATAMART}/bronze/{name}"))
    print(f"{name}: {len(files)} partitions, {files[0]} to {files[-1]}")

attributes: 25 partitions, bronze_attributes_2023_01_01.csv to bronze_attributes_2025_01_01.csv
financials: 25 partitions, bronze_financials_2023_01_01.csv to bronze_financials_2025_01_01.csv
clickstream: 24 partitions, bronze_clickstream_2023_01_01.csv to bronze_clickstream_2024_12_01.csv
loan_daily: 35 partitions, bronze_loan_daily_2023_01_01.csv to bronze_loan_daily_2025_11_01.csv


In [3]:
for name, file_name in SOURCES.items():
    raw = spark.read.csv(f"{DATA_DIR}/{file_name}", header=True, escape='"')
    bronze = spark.read.csv(f"{DATAMART}/bronze/{name}/*", header=True, escape='"')
    print(f"{name}: raw {raw.count()} rows, bronze {bronze.count()} rows, "
          f"raw rows not in bronze {raw.exceptAll(bronze).count()}, "
          f"bronze rows not in raw {bronze.exceptAll(raw).count()}")

attributes: raw 12500 rows, bronze 12500 rows, raw rows not in bronze 0, bronze rows not in raw 0
financials: raw 12500 rows, bronze 12500 rows, raw rows not in bronze 0, bronze rows not in raw 0
clickstream: raw 215376 rows, bronze 215376 rows, raw rows not in bronze 0, bronze rows not in raw 0
loan_daily: raw 137500 rows, bronze 137500 rows, raw rows not in bronze 0, bronze rows not in raw 0


**Findings:**
- Each source is stored as one partition per month
- Every raw row is in bronze and bronze has no other rows (0 differences in both directions), so bronze matches the raw data